# 05 — Optimisation du modèle ExpenseAI — V2 ciblée

## Objectif et règle d'isolation

Ce notebook optimise les hyperparamètres, compare plusieurs stratégies de gestion du déséquilibre et étudie le seuil de classification. La classe positive reste **Refusée = 1**.

La V2 ajoute une recherche ciblée pour la **régression logistique avec SMOTE**, en optimisant conjointement `C` et `k_neighbors`, puis la compare à une régression logistique avec `class_weight="balanced"`. Les décisions reposent exclusivement sur le train et sur des prédictions out-of-fold groupées.

Le jeu de test externe est recréé pour garantir la continuité avec `04_modeling.ipynb`, mais il n'est pas utilisé pendant l'optimisation. Cette étape ne réalise ni évaluation finale sur le test, ni SHAP, ni sauvegarde du modèle, ni écriture PostgreSQL, ni intégration Streamlit.

## 1. Imports et reproductibilité

In [21]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from sqlalchemy import text

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbalancedPipeline
from imblearn.under_sampling import RandomUnderSampler
from joblib import parallel_backend
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
pio.templates.default = "plotly_white"

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from database.connection import create_db_engine

## 2. Chargement depuis PostgreSQL

In [22]:
engine = create_db_engine()
try:
    with engine.connect() as connection:
        df = pd.read_sql(text("SELECT * FROM v_ml_expenses"), connection)
finally:
    engine.dispose()

numeric_columns_from_sql = [
    "amount_ttc",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
    "target",
]
for column in numeric_columns_from_sql:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["billable"] = df["billable"].astype(int)

target_counts = df["target"].value_counts().sort_index()
controls = pd.Series(
    {
        "Lignes": len(df),
        "Approuvées": int(target_counts.get(0, 0)),
        "Refusées": int(target_counts.get(1, 0)),
        "Valeurs manquantes": int(df.isna().sum().sum()),
        "Groupes": int(df["expense_group"].nunique()),
    },
    name="Valeur",
)
display(controls.to_frame())

assert len(df) == 7070
assert target_counts.to_dict() == {0: 6956, 1: 114}
assert set(df["target"].unique()) == {0, 1}
assert df.isna().sum().sum() == 0

,Valeur
Lignes,7070
Approuvées,6956
Refusées,114
Valeurs manquantes,0
Groupes,6253


## 3. Reproduction exacte de `X`, `y`, `groups` et du split externe

In [23]:
FEATURES = [
    "type",
    "amount_ttc",
    "billable",
    "project_code",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
]

X = df[FEATURES].copy()
y = df["target"].astype(int).copy()
groups = df["expense_group"].astype(str).copy()

outer_split = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
train_indices, test_indices = next(outer_split.split(X, y, groups=groups))

X_train = X.iloc[train_indices].copy()
y_train = y.iloc[train_indices].copy()
groups_train = groups.iloc[train_indices].copy()

# Le test est créé pour reproduire exactement le notebook 04, puis il est gelé.
X_test = X.iloc[test_indices].copy()
y_test = y.iloc[test_indices].copy()
groups_test = groups.iloc[test_indices].copy()

assert set(groups_train).isdisjoint(set(groups_test))

split_control = pd.DataFrame(
    {
        "Jeu": ["Train d'optimisation", "Test externe gelé"],
        "Lignes": [len(train_indices), len(test_indices)],
        "Groupes": [groups_train.nunique(), groups_test.nunique()],
        "Refus": [int(y.iloc[train_indices].sum()), int(y.iloc[test_indices].sum())],
        "Taux de refus (%)": [
            y.iloc[train_indices].mean() * 100,
            y.iloc[test_indices].mean() * 100,
        ],
    }
)
display(split_control)

TEST_IS_FROZEN = True
print("Jeu de test gelé après reproduction du split :", TEST_IS_FROZEN)

,Jeu,Lignes,Groupes,Refus,Taux de refus (%)
0,Train d'optimisation,5656,5001,91,1.6089
1,Test externe gelé,1414,1252,23,1.6266


Jeu de test gelé après reproduction du split : True


> **Jeu de test gelé.** Après la cellule précédente, `X_test` et `y_test` ne sont plus consultés dans ce notebook : aucune prédiction, probabilité ou métrique de test n'est calculée. Le test est donc **non utilisé pendant l'optimisation**. L'évaluation finale appartient au futur notebook `06_final_model.ipynb`.

## 4. Validation croisée interne et preprocessing

In [24]:
inner_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

CATEGORICAL_FEATURES = ["type", "project_code"]
NUMERIC_FEATURES = [
    "amount_ttc",
    "billable",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
]


def build_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    """Crée un preprocessing appris uniquement au sein de chaque fold."""
    numeric_transformer = (
        Pipeline([("standardisation", StandardScaler())])
        if scale_numeric
        else "passthrough"
    )
    return ColumnTransformer(
        transformers=[
            (
                "categories",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_FEATURES,
            ),
            ("numeriques", numeric_transformer, NUMERIC_FEATURES),
        ],
        remainder="drop",
    )


f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    pos_label=1,
    zero_division=0,
)
scoring = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "f1": make_scorer(f1_score, pos_label=1, zero_division=0),
    "f2": f2_scorer,
    "balanced_accuracy": "balanced_accuracy",
}

display(
    pd.DataFrame(
        {
            "Famille": ["Catégorielles", "Numériques"],
            "Variables": [", ".join(CATEGORICAL_FEATURES), ", ".join(NUMERIC_FEATURES)],
            "Traitement": [
                "OneHotEncoder(handle_unknown='ignore')",
                "StandardScaler pour la régression logistique ; passthrough pour les arbres",
            ],
        }
    )
)

,Famille,Variables,Traitement
0,Catégorielles,"type, project_code",OneHotEncoder(handle_unknown='ignore')
1,Numériques,"amount_ttc, billable, tax_rate, annee, mois, j...",StandardScaler pour la régression logistique ;...


La métrique de refit est l'**Average Precision (PR-AUC)**, adaptée à la rareté des refus. Le F2 complète l'analyse en accordant deux fois plus d'importance au recall qu'à la precision. Le preprocessing reste dans chaque pipeline : il n'est jamais ajusté avant la validation croisée.

## 5. Fonctions de synthèse des recherches

In [25]:
def extract_grid_result(model_name: str, search: GridSearchCV) -> dict:
    """Extrait les scores du meilleur candidat sans accéder au test externe."""
    index = search.best_index_
    results = search.cv_results_
    return {
        "Modèle": model_name,
        "Meilleurs paramètres": search.best_params_,
        "PR-AUC CV": results["mean_test_average_precision"][index],
        "Écart-type PR-AUC": results["std_test_average_precision"][index],
        "Recall CV": results["mean_test_recall"][index],
        "Precision CV": results["mean_test_precision"][index],
        "F1 CV": results["mean_test_f1"][index],
        "F2 CV": results["mean_test_f2"][index],
        "Balanced Accuracy CV": results["mean_test_balanced_accuracy"][index],
        "ROC-AUC CV": results["mean_test_roc_auc"][index],
        "PR-AUC train": results["mean_train_average_precision"][index],
        "F2 train": results["mean_train_f2"][index],
        "Écart train-validation": (
            results["mean_train_average_precision"][index]
            - results["mean_test_average_precision"][index]
        ),
    }


def run_grid_search(pipeline: Pipeline, parameter_grid: dict) -> GridSearchCV:
    """Exécute une recherche groupée exclusivement sur le train."""
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring=scoring,
        refit="average_precision",
        cv=inner_cv,
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
    )
    # Le backend threads évite un défaut du resource tracker avec Python 3.14,
    # tout en conservant n_jobs=-1 pour paralléliser la recherche.
    with parallel_backend("threading"):
        search.fit(X_train, y_train, groups=groups_train)
    return search

## 6. GridSearch — Régression logistique

In [26]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessing", build_preprocessor(scale_numeric=True)),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
logistic_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__class_weight": ["balanced"],
}
logistic_search = run_grid_search(logistic_pipeline, logistic_grid)
logistic_result = extract_grid_result("Régression logistique", logistic_search)
display(pd.Series(logistic_result, name="Valeur").to_frame())

,Valeur
Modèle,Régression logistique
Meilleurs paramètres,"{'model__C': 1, 'model__class_weight': 'balanc..."
PR-AUC CV,0.1018
Écart-type PR-AUC,0.0404
Recall CV,0.6246
Precision CV,0.0528
F1 CV,0.0973
F2 CV,0.1969
Balanced Accuracy CV,0.7208
ROC-AUC CV,0.7795


## 7. GridSearch — Decision Tree

In [27]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", build_preprocessor(scale_numeric=False)),
        (
            "model",
            DecisionTreeClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
tree_grid = {
    "model__max_depth": [3, 5, 8, 12, None],
    "model__min_samples_split": [2, 10, 30],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__criterion": ["gini", "entropy"],
    "model__class_weight": ["balanced"],
}
tree_search = run_grid_search(tree_pipeline, tree_grid)
tree_result = extract_grid_result("Arbre de décision", tree_search)
display(pd.Series(tree_result, name="Valeur").to_frame())

,Valeur
Modèle,Arbre de décision
Meilleurs paramètres,"{'model__class_weight': 'balanced', 'model__cr..."
PR-AUC CV,0.1450
Écart-type PR-AUC,0.0658
Recall CV,0.3842
Precision CV,0.0411
F1 CV,0.0728
F2 CV,0.1371
Balanced Accuracy CV,0.5968
ROC-AUC CV,0.6274


## 8. GridSearch — Random Forest

In [28]:
forest_pipeline = Pipeline(
    steps=[
        ("preprocessing", build_preprocessor(scale_numeric=False)),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)
forest_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 5, 10],
    "model__max_features": ["sqrt"],
    "model__class_weight": ["balanced_subsample"],
}
forest_search = run_grid_search(forest_pipeline, forest_grid)
forest_result = extract_grid_result("Forêt aléatoire", forest_search)
display(pd.Series(forest_result, name="Valeur").to_frame())

,Valeur
Modèle,Forêt aléatoire
Meilleurs paramètres,"{'model__class_weight': 'balanced_subsample', ..."
PR-AUC CV,0.1545
Écart-type PR-AUC,0.0494
Recall CV,0.0550
Precision CV,0.6000
F1 CV,0.1000
F2 CV,0.0670
Balanced Accuracy CV,0.5273
ROC-AUC CV,0.7895


## 9. Tableau des GridSearch

In [29]:
grid_summary = pd.DataFrame(
    [logistic_result, tree_result, forest_result]
).sort_values("PR-AUC CV", ascending=False)
display(grid_summary)

,Modèle,Meilleurs paramètres,PR-AUC CV,Écart-type PR-AUC,Recall CV,Precision CV,F1 CV,F2 CV,Balanced Accuracy CV,ROC-AUC CV,PR-AUC train,F2 train,Écart train-validation
2,Forêt aléatoire,"{'model__class_weight': 'balanced_subsample', ...",0.1545,0.0494,0.0550,0.6000,0.1000,0.0670,0.5273,0.7895,0.9973,0.9978,0.8428
1,Arbre de décision,"{'model__class_weight': 'balanced', 'model__cr...",0.1450,0.0658,0.3842,0.0411,0.0728,0.1371,0.5968,0.6274,0.3394,0.2717,0.1945
0,Régression logistique,"{'model__C': 1, 'model__class_weight': 'balanc...",0.1018,0.0404,0.6246,0.0528,0.0973,0.1969,0.7208,0.7795,0.1368,0.2740,0.0350


## 10. Analyse du surapprentissage avant/après optimisation

In [30]:
baseline_models = {
    "Arbre de décision": Pipeline(
        [
            ("preprocessing", build_preprocessor(scale_numeric=False)),
            (
                "model",
                DecisionTreeClassifier(
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Forêt aléatoire": Pipeline(
        [
            ("preprocessing", build_preprocessor(scale_numeric=False)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    ),
}

overfit_rows = []
optimized_lookup = {
    "Arbre de décision": tree_result,
    "Forêt aléatoire": forest_result,
}
for model_name, baseline_pipeline in baseline_models.items():
    with parallel_backend("threading"):
        baseline_scores = cross_validate(
            baseline_pipeline,
            X_train,
            y_train,
            groups=groups_train,
            cv=inner_cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=-1,
            error_score="raise",
        )
    baseline_train_pr = baseline_scores["train_average_precision"].mean()
    baseline_validation_pr = baseline_scores["test_average_precision"].mean()
    baseline_train_f2 = baseline_scores["train_f2"].mean()
    baseline_validation_f2 = baseline_scores["test_f2"].mean()
    optimized = optimized_lookup[model_name]
    overfit_rows.extend(
        [
            {
                "Modèle": model_name,
                "Configuration": "Initiale 04_modeling",
                "PR-AUC train": baseline_train_pr,
                "PR-AUC validation": baseline_validation_pr,
                "Écart PR-AUC": baseline_train_pr - baseline_validation_pr,
                "F2 train": baseline_train_f2,
                "F2 validation": baseline_validation_f2,
                "Écart F2": baseline_train_f2 - baseline_validation_f2,
            },
            {
                "Modèle": model_name,
                "Configuration": "Optimisée",
                "PR-AUC train": optimized["PR-AUC train"],
                "PR-AUC validation": optimized["PR-AUC CV"],
                "Écart PR-AUC": optimized["Écart train-validation"],
                "F2 train": optimized["F2 train"],
                "F2 validation": optimized["F2 CV"],
                "Écart F2": optimized["F2 train"] - optimized["F2 CV"],
            },
        ]
    )

overfit_comparison = pd.DataFrame(overfit_rows)
display(overfit_comparison)

gap_changes = (
    overfit_comparison.pivot(index="Modèle", columns="Configuration", values="Écart PR-AUC")
    .assign(
        reduction=lambda table: table["Initiale 04_modeling"] - table["Optimisée"]
    )
    .reset_index()
)
display(gap_changes)

,Modèle,Configuration,PR-AUC train,PR-AUC validation,Écart PR-AUC,F2 train,F2 validation,Écart F2
0,Arbre de décision,Initiale 04_modeling,0.9999,0.0520,0.9478,0.9978,0.1687,0.8291
1,Arbre de décision,Optimisée,0.3394,0.1450,0.1945,0.2717,0.1371,0.1346
2,Forêt aléatoire,Initiale 04_modeling,0.9961,0.1493,0.8468,0.9978,0.0672,0.9306
3,Forêt aléatoire,Optimisée,0.9973,0.1545,0.8428,0.9978,0.0670,0.9308


Configuration,Modèle,Initiale 04_modeling,Optimisée,reduction
0,Arbre de décision,0.9478,0.1945,0.7533
1,Forêt aléatoire,0.8468,0.8428,0.0040


Une réduction positive de l'écart indique que la configuration optimisée rapproche les performances train et validation. L'analyse combine PR-AUC et F2 : un seul indicateur ne suffit pas à diagnostiquer le surapprentissage.

## 11. V2 ciblée — Logistic Regression avec `class_weight` ou SMOTE

Les recherches précédentes sur l'arbre et la forêt sont conservées comme éléments de comparaison. La V2 ne les recommence pas : elle approfondit uniquement la régression logistique, qui offrait le meilleur profil de détection.

Deux recherches utilisent exactement les mêmes folds groupés :

- `class_weight="balanced"` avec optimisation de `C` ;
- SMOTE dans le pipeline avec optimisation conjointe de `C` et `k_neighbors`.

Le rééchantillonnage reste strictement limité aux sous-échantillons d'entraînement de la validation croisée.

In [31]:
logistic_c_values = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

balanced_v2_pipeline = Pipeline(
    steps=[
        ("preprocessing", build_preprocessor(scale_numeric=True)),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
balanced_v2_search = run_grid_search(
    balanced_v2_pipeline,
    {
        "model__C": logistic_c_values,
        "model__penalty": ["l2"],
    },
)

smote_v2_pipeline = ImbalancedPipeline(
    steps=[
        ("preprocessing", build_preprocessor(scale_numeric=True)),
        ("sampling", SMOTE(random_state=RANDOM_STATE)),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
smote_v2_search = run_grid_search(
    smote_v2_pipeline,
    {
        "model__C": logistic_c_values,
        "model__penalty": ["l2"],
        "sampling__k_neighbors": [2, 3, 5, 7, 10],
    },
)

balanced_v2_result = extract_grid_result(
    "Logistic Regression + class_weight", balanced_v2_search
)
smote_v2_result = extract_grid_result(
    "Logistic Regression + SMOTE", smote_v2_search
)

v2_comparison = pd.DataFrame(
    [balanced_v2_result, smote_v2_result]
).sort_values("PR-AUC CV", ascending=False)

# Règle prudente définie avant le choix : le gain de SMOTE doit être matériel
# et reproductible sur au moins quatre des cinq folds groupés.
SMOTE_MIN_PR_AUC_GAIN = 0.01
SMOTE_MIN_WINNING_FOLDS = 4

balanced_index = balanced_v2_search.best_index_
smote_index = smote_v2_search.best_index_
balanced_fold_scores = np.array(
    [
        balanced_v2_search.cv_results_[f"split{fold}_test_average_precision"][
            balanced_index
        ]
        for fold in range(inner_cv.n_splits)
    ]
)
smote_fold_scores = np.array(
    [
        smote_v2_search.cv_results_[f"split{fold}_test_average_precision"][
            smote_index
        ]
        for fold in range(inner_cv.n_splits)
    ]
)
paired_pr_auc_gains = smote_fold_scores - balanced_fold_scores
mean_pr_auc_gain = float(
    smote_v2_result["PR-AUC CV"] - balanced_v2_result["PR-AUC CV"]
)
winning_folds = int((paired_pr_auc_gains > 0).sum())
smote_gain_convincing = (
    mean_pr_auc_gain >= SMOTE_MIN_PR_AUC_GAIN
    and winning_folds >= SMOTE_MIN_WINNING_FOLDS
)

selected_strategy = "SMOTE" if smote_gain_convincing else "class_weight"
selected_search = smote_v2_search if smote_gain_convincing else balanced_v2_search
selected_v2_result = smote_v2_result if smote_gain_convincing else balanced_v2_result

display(v2_comparison)
display(
    pd.DataFrame(
        {
            "Fold groupé": np.arange(1, inner_cv.n_splits + 1),
            "PR-AUC class_weight": balanced_fold_scores,
            "PR-AUC SMOTE": smote_fold_scores,
            "Gain SMOTE": paired_pr_auc_gains,
        }
    )
)
display(
    pd.Series(
        {
            "Gain moyen de PR-AUC avec SMOTE": mean_pr_auc_gain,
            "Folds gagnés par SMOTE": f"{winning_folds}/{inner_cv.n_splits}",
            "Gain minimal exigé": SMOTE_MIN_PR_AUC_GAIN,
            "Folds gagnants exigés": (
                f"{SMOTE_MIN_WINNING_FOLDS}/{inner_cv.n_splits}"
            ),
            "Gain jugé suffisamment convaincant": smote_gain_convincing,
            "Stratégie retenue pour la suite": selected_strategy,
        },
        name="Décision V2",
    ).to_frame()
)

,Modèle,Meilleurs paramètres,PR-AUC CV,Écart-type PR-AUC,Recall CV,Precision CV,F1 CV,F2 CV,Balanced Accuracy CV,ROC-AUC CV,PR-AUC train,F2 train,Écart train-validation
1,Logistic Regression + SMOTE,"{'model__C': 1, 'model__penalty': 'l2', 'sampl...",0.1093,0.0449,0.5918,0.0550,0.1006,0.1999,0.7126,0.7703,0.1441,0.2859,0.0347
0,Logistic Regression + class_weight,"{'model__C': 1, 'model__penalty': 'l2'}",0.1018,0.0404,0.6246,0.0528,0.0973,0.1969,0.7208,0.7795,0.1368,0.2740,0.0350


,Fold groupé,PR-AUC class_weight,PR-AUC SMOTE,Gain SMOTE
0,1,0.0709,0.0752,0.0043
1,2,0.1005,0.1064,0.0059
2,3,0.0685,0.0762,0.0077
3,4,0.0900,0.0929,0.0029
4,5,0.1790,0.1960,0.0170


,Décision V2
Gain moyen de PR-AUC avec SMOTE,0.0076
Folds gagnés par SMOTE,5/5
Gain minimal exigé,0.0100
Folds gagnants exigés,4/5
Gain jugé suffisamment convaincant,False
Stratégie retenue pour la suite,class_weight


### Règle de décision et limite méthodologique de SMOTE

SMOTE n'est retenu que si son gain moyen de PR-AUC atteint au moins **0,01 point absolu** et s'il améliore la PR-AUC dans au moins **4 folds sur 5** face à `class_weight="balanced"`. Cette règle prudente évite de choisir une méthode plus complexe pour un gain faible ou instable.

SMOTE est appliqué **après** le `OneHotEncoder` et uniquement dans le fold d'entraînement grâce au pipeline `imblearn`. Il n'y a donc pas de fuite entre validation et entraînement. Cependant, les interpolations dans un espace one-hot peuvent créer des combinaisons fractionnaires moins naturelles pour les catégories. La comparaison doit ainsi favoriser la solution simple si le bénéfice n'est pas suffisamment convaincant.

## 12. Configuration logistique retenue pour l'étude des seuils

In [32]:
def build_v2_logistic_pipeline(strategy: str) -> ImbalancedPipeline:
    """Reconstruit la meilleure configuration V2 sans utiliser le test externe."""
    if strategy == "SMOTE":
        parameters = smote_v2_search.best_params_
        return ImbalancedPipeline(
            steps=[
                ("preprocessing", build_preprocessor(scale_numeric=True)),
                (
                    "sampling",
                    SMOTE(
                        random_state=RANDOM_STATE,
                        k_neighbors=parameters["sampling__k_neighbors"],
                    ),
                ),
                (
                    "model",
                    LogisticRegression(
                        C=parameters["model__C"],
                        penalty=parameters["model__penalty"],
                        class_weight=None,
                        max_iter=2000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )

    parameters = balanced_v2_search.best_params_
    return ImbalancedPipeline(
        steps=[
            ("preprocessing", build_preprocessor(scale_numeric=True)),
            (
                "model",
                LogisticRegression(
                    C=parameters["model__C"],
                    penalty=parameters["model__penalty"],
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


candidate_configurations = pd.DataFrame(
    [
        {
            "Stratégie": "class_weight",
            "Paramètres": balanced_v2_search.best_params_,
            "PR-AUC CV": balanced_v2_result["PR-AUC CV"],
            "F2 CV à 0,5": balanced_v2_result["F2 CV"],
            "Retenue": selected_strategy == "class_weight",
        },
        {
            "Stratégie": "SMOTE",
            "Paramètres": smote_v2_search.best_params_,
            "PR-AUC CV": smote_v2_result["PR-AUC CV"],
            "F2 CV à 0,5": smote_v2_result["F2 CV"],
            "Retenue": selected_strategy == "SMOTE",
        },
    ]
)
display(candidate_configurations)

decision_text = (
    "Le gain de SMOTE satisfait la règle prudente ; SMOTE est retenu pour "
    "l'étude des seuils."
    if smote_gain_convincing
    else "Le gain de SMOTE ne satisfait pas la règle prudente ; "
    "`class_weight=\"balanced\"` est conservé pour sa simplicité et sa robustesse."
)
display(Markdown(decision_text))

,Stratégie,Paramètres,PR-AUC CV,"F2 CV à 0,5",Retenue
0,class_weight,"{'model__C': 1, 'model__penalty': 'l2'}",0.1018,0.1969,True
1,SMOTE,"{'model__C': 1, 'model__penalty': 'l2', 'sampl...",0.1093,0.1999,False


Le gain de SMOTE ne satisfait pas la règle prudente ; `class_weight="balanced"` est conservé pour sa simplicité et sa robustesse.

## 13. Probabilités out-of-fold sur le train

Les deux stratégies sont évaluées sur les mêmes folds pour documenter leur comportement. Seules les probabilités out-of-fold du train sont utilisées ; le jeu de test n'intervient pas.

In [33]:
candidate_oof_probabilities = {}
for strategy in ["class_weight", "SMOTE"]:
    with parallel_backend("threading"):
        probabilities = cross_val_predict(
            build_v2_logistic_pipeline(strategy),
            X_train,
            y_train,
            groups=groups_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1,
        )[:, 1]
    assert len(probabilities) == len(y_train)
    assert np.isfinite(probabilities).all()
    candidate_oof_probabilities[strategy] = probabilities

oof_probabilities = candidate_oof_probabilities[selected_strategy]
print(
    "Probabilités out-of-fold calculées sur le train pour :",
    ", ".join(candidate_oof_probabilities),
)
print("Stratégie utilisée pour l'étude détaillée des seuils :", selected_strategy)

Probabilités out-of-fold calculées sur le train pour : class_weight, SMOTE
Stratégie utilisée pour l'étude détaillée des seuils : class_weight


## 14. Métriques selon le seuil

In [34]:
def calculate_threshold_metrics(probabilities: np.ndarray) -> pd.DataFrame:
    """Calcule les métriques out-of-fold pour chaque seuil testé."""
    threshold_rows = []
    for threshold in np.round(np.arange(0.05, 0.96, 0.01), 2):
        predictions = (probabilities >= threshold).astype(int)
        true_negative, false_positive, false_negative, true_positive = confusion_matrix(
            y_train,
            predictions,
            labels=[0, 1],
        ).ravel()
        threshold_rows.append(
            {
                "Threshold": threshold,
                "Precision": precision_score(y_train, predictions, zero_division=0),
                "Recall": recall_score(y_train, predictions, zero_division=0),
                "F1": f1_score(y_train, predictions, zero_division=0),
                "F2": fbeta_score(y_train, predictions, beta=2, zero_division=0),
                "Balanced Accuracy": balanced_accuracy_score(y_train, predictions),
                "True Negatives": int(true_negative),
                "False Positives": int(false_positive),
                "False Negatives": int(false_negative),
                "True Positives": int(true_positive),
            }
        )
    return pd.DataFrame(threshold_rows)


threshold_metrics = calculate_threshold_metrics(oof_probabilities)
scenario_f1 = threshold_metrics.sort_values(
    ["F1", "Precision", "Threshold"],
    ascending=[False, False, False],
).iloc[0]
scenario_f2 = threshold_metrics.sort_values(
    ["F2", "Recall", "Precision"],
    ascending=[False, False, False],
).iloc[0]
recall_candidates = threshold_metrics[threshold_metrics["Recall"] >= 0.70]
if recall_candidates.empty:
    scenario_recall = None
    recall_rule = "Aucun seuil testé n'atteint un recall de 70 %"
else:
    scenario_recall = recall_candidates.sort_values(
        ["Precision", "F2", "Threshold"],
        ascending=[False, False, False],
    ).iloc[0]
    recall_rule = "Recall ≥ 70 %, puis meilleure precision"

scenario_rows = []
for label, row in [
    ("Référence — seuil 0,50", threshold_metrics.loc[(threshold_metrics["Threshold"] - 0.50).abs().idxmin()]),
    ("Équilibre — F1 maximal", scenario_f1),
    ("Détection — F2 maximal", scenario_f2),
    ("Métier — recall ≥ 70 %", scenario_recall),
]:
    if row is None:
        continue
    values = row.to_dict()
    values["Scénario"] = label
    scenario_rows.append(values)

threshold_candidates = (
    pd.DataFrame(scenario_rows)
    .drop_duplicates(subset=["Threshold"])
    .reset_index(drop=True)
)
display(
    threshold_candidates[
        [
            "Scénario",
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "F2",
            "Balanced Accuracy",
            "False Positives",
            "False Negatives",
        ]
    ]
)

,Scénario,Threshold,Precision,Recall,F1,F2,Balanced Accuracy,False Positives,False Negatives
0,"Référence — seuil 0,50",0.5000,0.0530,0.6264,0.0978,0.1981,0.7217,"1,018.0000",34.0000
1,Équilibre — F1 maximal,0.8700,0.1694,0.3407,0.2263,0.2834,0.6567,152.0000,60.0000
2,Métier — recall ≥ 70 %,0.3400,0.0395,0.7033,0.0749,0.1614,0.7119,"1,555.0000",27.0000


In [35]:
metric_figure = go.Figure()
for metric, color in [
    ("Precision", "#0F766E"),
    ("Recall", "#DC2626"),
    ("F1", "#2563EB"),
    ("F2", "#7C3AED"),
]:
    metric_figure.add_trace(
        go.Scatter(
            x=threshold_metrics["Threshold"],
            y=threshold_metrics[metric],
            mode="lines",
            name=metric,
            line={"color": color},
        )
    )
metric_figure.update_layout(
    title="Precision, recall, F1 et F2 selon le seuil — prédictions out-of-fold",
    xaxis_title="Seuil",
    yaxis_title="Score",
    hovermode="x unified",
)
metric_figure.show()

error_figure = go.Figure()
error_figure.add_trace(
    go.Scatter(
        x=threshold_metrics["Threshold"],
        y=threshold_metrics["False Positives"],
        mode="lines",
        name="Faux positifs",
        line={"color": "#F59E0B"},
    )
)
error_figure.add_trace(
    go.Scatter(
        x=threshold_metrics["Threshold"],
        y=threshold_metrics["False Negatives"],
        mode="lines",
        name="Faux négatifs",
        line={"color": "#DC2626"},
    )
)
error_figure.update_layout(
    title="Faux positifs et faux négatifs selon le seuil",
    xaxis_title="Seuil",
    yaxis_title="Nombre de lignes",
    hovermode="x unified",
)
error_figure.show()

## 15. Plusieurs seuils candidats — aucun seuil final à ce stade

In [36]:
display(
    threshold_candidates[
        [
            "Scénario",
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "F2",
            "False Positives",
            "False Negatives",
        ]
    ]
)

print("Configuration V2 retenue pour comparaison future : Logistic Regression /", selected_strategy)
print("Seuils candidats conservés :", threshold_candidates["Threshold"].tolist())
print("Aucun seuil n'est déclaré final dans ce notebook.")

,Scénario,Threshold,Precision,Recall,F1,F2,False Positives,False Negatives
0,"Référence — seuil 0,50",0.5000,0.0530,0.6264,0.0978,0.1981,"1,018.0000",34.0000
1,Équilibre — F1 maximal,0.8700,0.1694,0.3407,0.2263,0.2834,152.0000,60.0000
2,Métier — recall ≥ 70 %,0.3400,0.0395,0.7033,0.0749,0.1614,"1,555.0000",27.0000


Configuration V2 retenue pour comparaison future : Logistic Regression / class_weight
Seuils candidats conservés : [0.5, 0.87, 0.34]
Aucun seuil n'est déclaré final dans ce notebook.


## 16. Comparaison des seuils candidats

In [37]:
threshold_comparison = threshold_candidates[
    [
        "Scénario",
        "Threshold",
        "Precision",
        "Recall",
        "F1",
        "F2",
        "Balanced Accuracy",
        "False Positives",
        "False Negatives",
    ]
].copy()
display(threshold_comparison)

,Scénario,Threshold,Precision,Recall,F1,F2,Balanced Accuracy,False Positives,False Negatives
0,"Référence — seuil 0,50",0.5000,0.0530,0.6264,0.0978,0.1981,0.7217,"1,018.0000",34.0000
1,Équilibre — F1 maximal,0.8700,0.1694,0.3407,0.2263,0.2834,0.6567,152.0000,60.0000
2,Métier — recall ≥ 70 %,0.3400,0.0395,0.7033,0.0749,0.1614,0.7119,"1,555.0000",27.0000


Les seuils sont conservés comme **scénarios candidats**, et non comme décision finale :

- `0,50` fournit un point de référence reproductible ;
- le seuil de F1 recherche un équilibre entre precision et recall ;
- le seuil de F2 privilégie davantage la détection des refus ;
- le seuil métier vise au moins 70 % de recall lorsqu'un tel seuil existe.

Le choix définitif devra être fait dans `06_final_model.ipynb` à partir d'un coût métier explicite des faux positifs et faux négatifs. En particulier, appeler `0,27` « seuil final » serait prématuré : ce seuil correspond seulement à un scénario de forte détection sur les prédictions out-of-fold du train.

## 17. Effet des seuils candidats sur la matrice de confusion

In [38]:
confusion_rows = []
for _, candidate in threshold_candidates.iterrows():
    predictions = (oof_probabilities >= candidate["Threshold"]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, predictions, labels=[0, 1]).ravel()
    confusion_rows.append(
        {
            "Scénario": candidate["Scénario"],
            "Seuil": candidate["Threshold"],
            "Vrais négatifs": int(tn),
            "Faux positifs": int(fp),
            "Faux négatifs": int(fn),
            "Vrais positifs": int(tp),
        }
    )

confusion_candidates = pd.DataFrame(confusion_rows)
display(confusion_candidates)

confusion_figure = go.Figure()
for error_name, color in [
    ("Faux positifs", "#F59E0B"),
    ("Faux négatifs", "#DC2626"),
]:
    confusion_figure.add_trace(
        go.Bar(
            x=confusion_candidates["Scénario"],
            y=confusion_candidates[error_name],
            name=error_name,
            marker_color=color,
        )
    )
confusion_figure.update_layout(
    title="Erreurs out-of-fold selon les seuils candidats",
    xaxis_title="Scénario",
    yaxis_title="Nombre de lignes",
    barmode="group",
)
confusion_figure.show()

,Scénario,Seuil,Vrais négatifs,Faux positifs,Faux négatifs,Vrais positifs
0,"Référence — seuil 0,50",0.5000,4547,1018,34,57
1,Équilibre — F1 maximal,0.8700,5413,152,60,31
2,Métier — recall ≥ 70 %,0.3400,4010,1555,27,64


## 18. Configuration V2 proposée — train uniquement

In [39]:
if selected_strategy == "SMOTE":
    selected_parameters = {
        "C": smote_v2_search.best_params_["model__C"],
        "penalty": smote_v2_search.best_params_["model__penalty"],
        "class_weight": None,
        "k_neighbors": smote_v2_search.best_params_["sampling__k_neighbors"],
    }
else:
    selected_parameters = {
        "C": balanced_v2_search.best_params_["model__C"],
        "penalty": balanced_v2_search.best_params_["model__penalty"],
        "class_weight": "balanced",
        "k_neighbors": "sans objet",
    }

provisional_configuration = pd.Series(
    {
        "Algorithme": "Logistic Regression",
        "Hyperparamètres": selected_parameters,
        "Stratégie de déséquilibre": selected_strategy,
        "Gain PR-AUC de SMOTE": mean_pr_auc_gain,
        "SMOTE suffisamment convaincant": smote_gain_convincing,
        "Métrique principale": "Average Precision / PR-AUC",
        "PR-AUC CV": selected_v2_result["PR-AUC CV"],
        "Recall CV à 0,5": selected_v2_result["Recall CV"],
        "F2 CV à 0,5": selected_v2_result["F2 CV"],
        "Seuils candidats": threshold_candidates["Threshold"].tolist(),
        "Seuil final": "non défini à ce stade",
    },
    name="Configuration V2",
)
display(provisional_configuration.to_frame())

assert TEST_IS_FROZEN
print("Le test externe n'a pas été utilisé pendant l'optimisation.")

,Configuration V2
Algorithme,Logistic Regression
Hyperparamètres,"{'C': 1, 'penalty': 'l2', 'class_weight': 'bal..."
Stratégie de déséquilibre,class_weight
Gain PR-AUC de SMOTE,0.0076
SMOTE suffisamment convaincant,False
Métrique principale,Average Precision / PR-AUC
PR-AUC CV,0.1018
"Recall CV à 0,5",0.6246
"F2 CV à 0,5",0.1969
Seuils candidats,"[0.5, 0.87, 0.34]"


Le test externe n'a pas été utilisé pendant l'optimisation.


## Synthèse de l'optimisation V2

In [40]:
def metric_fr(value: float) -> str:
    """Formate une métrique selon la notation française."""
    return f"{value:.4f}".replace(".", ",")


smote_decision_sentence = (
    "Le gain satisfait les deux critères : SMOTE est retenu."
    if smote_gain_convincing
    else "Le gain ne satisfait pas les deux critères : `class_weight=\"balanced\"` "
    "est retenu, sans ajouter la complexité de SMOTE."
)
threshold_lines = []
for _, row in threshold_candidates.iterrows():
    threshold_lines.append(
        f"- **{row['Scénario']}** — seuil {row['Threshold']:.2f}, "
        f"precision {metric_fr(row['Precision'])}, recall {metric_fr(row['Recall'])}, "
        f"F2 {metric_fr(row['F2'])}, {int(row['False Positives'])} faux positifs et "
        f"{int(row['False Negatives'])} faux négatifs."
    )

summary = f"""
### Comparaison ciblée Logistic Regression

- **class_weight="balanced"** — paramètres {balanced_v2_search.best_params_}, PR-AUC CV {metric_fr(balanced_v2_result['PR-AUC CV'])}, recall {metric_fr(balanced_v2_result['Recall CV'])}, F2 {metric_fr(balanced_v2_result['F2 CV'])}.
- **SMOTE optimisé** — paramètres {smote_v2_search.best_params_}, PR-AUC CV {metric_fr(smote_v2_result['PR-AUC CV'])}, recall {metric_fr(smote_v2_result['Recall CV'])}, F2 {metric_fr(smote_v2_result['F2 CV'])}.
- Le gain moyen de PR-AUC de SMOTE vaut **{mean_pr_auc_gain:+.4f}** et SMOTE gagne **{winning_folds}/{inner_cv.n_splits} folds**.
- La règle exige un gain d'au moins {SMOTE_MIN_PR_AUC_GAIN:.2f} et au moins {SMOTE_MIN_WINNING_FOLDS}/{inner_cv.n_splits} folds gagnants. {smote_decision_sentence}

### Seuils candidats conservés

{chr(10).join(threshold_lines)}

Aucun de ces seuils n'est déclaré final. Le seuil métier de forte détection reste une option coûteuse en faux positifs et devra être arbitré avec un coût métier explicite.

### Configuration V2 proposée

- **Algorithme :** Logistic Regression
- **Hyperparamètres :** {selected_parameters}
- **Stratégie de déséquilibre :** {selected_strategy}
- **Métrique principale :** Average Precision / PR-AUC
- **Seuil final :** non défini ; plusieurs scénarios sont transmis au notebook 06.

Cette sélection est fondée uniquement sur la validation groupée du train. Le faible nombre de refus rend les moyennes et les seuils sensibles à quelques observations. Le test externe est **non utilisé pendant l'optimisation**.
"""
display(Markdown(summary))


### Comparaison ciblée Logistic Regression

- **class_weight="balanced"** — paramètres {'model__C': 1, 'model__penalty': 'l2'}, PR-AUC CV 0,1018, recall 0,6246, F2 0,1969.
- **SMOTE optimisé** — paramètres {'model__C': 1, 'model__penalty': 'l2', 'sampling__k_neighbors': 3}, PR-AUC CV 0,1093, recall 0,5918, F2 0,1999.
- Le gain moyen de PR-AUC de SMOTE vaut **+0.0076** et SMOTE gagne **5/5 folds**.
- La règle exige un gain d'au moins 0.01 et au moins 4/5 folds gagnants. Le gain ne satisfait pas les deux critères : `class_weight="balanced"` est retenu, sans ajouter la complexité de SMOTE.

### Seuils candidats conservés

- **Référence — seuil 0,50** — seuil 0.50, precision 0,0530, recall 0,6264, F2 0,1981, 1018 faux positifs et 34 faux négatifs.
- **Équilibre — F1 maximal** — seuil 0.87, precision 0,1694, recall 0,3407, F2 0,2834, 152 faux positifs et 60 faux négatifs.
- **Métier — recall ≥ 70 %** — seuil 0.34, precision 0,0395, recall 0,7033, F2 0,1614, 1555 faux positifs et 27 faux négatifs.

Aucun de ces seuils n'est déclaré final. Le seuil métier de forte détection reste une option coûteuse en faux positifs et devra être arbitré avec un coût métier explicite.

### Configuration V2 proposée

- **Algorithme :** Logistic Regression
- **Hyperparamètres :** {'C': 1, 'penalty': 'l2', 'class_weight': 'balanced', 'k_neighbors': 'sans objet'}
- **Stratégie de déséquilibre :** class_weight
- **Métrique principale :** Average Precision / PR-AUC
- **Seuil final :** non défini ; plusieurs scénarios sont transmis au notebook 06.

Cette sélection est fondée uniquement sur la validation groupée du train. Le faible nombre de refus rend les moyennes et les seuils sensibles à quelques observations. Le test externe est **non utilisé pendant l'optimisation**.


## 20. Étape suivante

Le prochain notebook sera `06_final_model.ipynb`. Après verrouillage complet de la configuration et choix métier du seuil, il pourra comprendre :

1. l'entraînement final sur tout le train ;
2. une unique évaluation finale sur le test gelé ;
3. l'analyse SHAP ;
4. l'interprétation des variables ;
5. la sauvegarde contrôlée du pipeline avec joblib ;
6. la préparation de l'intégration Streamlit.

Aucune de ces opérations n'est réalisée ici. Le test a été **non utilisé pendant l'optimisation** de cette V2.